In [8]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [9]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    temperature=0
)

llm = ChatHuggingFace(llm=llm)

In [10]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [11]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [12]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [13]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [14]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke relies on a play on words, combining a common phrase associated with emotional states ("feeling a little crusty") with a literal characteristic of a pizza ("crusty"). \n\nIn everyday language, someone who is "feeling a little crusty" is usually expressing that they\'re annoyed, irritable, or experiencing stress. However, in the context of a pizza, the crust is the outer layer, typically crispy and hard. \n\nBy using this wordplay, the joke creates a pun that connects the emotional state of the pizza to its literal crust. This unexpected twist on meaning creates the humor, allowing the listener to experience a brief moment of mental processing and then arrive at the punchline, where the connection between the emotional state and the crust becomes clear.'}

In [15]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke relies on a play on words, combining a common phrase associated with emotional states ("feeling a little crusty") with a literal characteristic of a pizza ("crusty"). \n\nIn everyday language, someone who is "feeling a little crusty" is usually expressing that they\'re annoyed, irritable, or experiencing stress. However, in the context of a pizza, the crust is the outer layer, typically crispy and hard. \n\nBy using this wordplay, the joke creates a pun that connects the emotional state of the pizza to its literal crust. This unexpected twist on meaning creates the humor, allowing the listener to experience a brief moment of mental processing and then arrive at the punchline, where the connection between the emotional state and the crust becomes clear.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpo

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke relies on a play on words, combining a common phrase associated with emotional states ("feeling a little crusty") with a literal characteristic of a pizza ("crusty"). \n\nIn everyday language, someone who is "feeling a little crusty" is usually expressing that they\'re annoyed, irritable, or experiencing stress. However, in the context of a pizza, the crust is the outer layer, typically crispy and hard. \n\nBy using this wordplay, the joke creates a pun that connects the emotional state of the pizza to its literal crust. This unexpected twist on meaning creates the humor, allowing the listener to experience a brief moment of mental processing and then arrive at the punchline, where the connection between the emotional state and the crust becomes clear.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkp

In [17]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy? \n\nBecause it was feeling a little "drained" and had a lot of "twisted" emotions.',
 'explanation': 'The joke relies on a play on words, using common phrases related to emotional states and plumbing to create a pun. \n\n"Feeling a little \'drained\'" refers to the idea that the spaghetti is physically drained of its liquid sauce, but also emotionally drained, which is a common phrase to describe someone feeling exhausted or depleted. \n\n"Twisted emotions" is a double entendre, referencing both the fact that spaghetti is twisted into strands and the emotional state of being mentally conflicted or having mixed feelings. \n\nThe joke relies on the listener being familiar with the common phrases and applying them to the setting of a spaghetti going to therapy, creating a humorous and unexpected connection between the two.'}

In [18]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke relies on a play on words, combining a common phrase associated with emotional states ("feeling a little crusty") with a literal characteristic of a pizza ("crusty"). \n\nIn everyday language, someone who is "feeling a little crusty" is usually expressing that they\'re annoyed, irritable, or experiencing stress. However, in the context of a pizza, the crust is the outer layer, typically crispy and hard. \n\nBy using this wordplay, the joke creates a pun that connects the emotional state of the pizza to its literal crust. This unexpected twist on meaning creates the humor, allowing the listener to experience a brief moment of mental processing and then arrive at the punchline, where the connection between the emotional state and the crust becomes clear.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpo

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke relies on a play on words, combining a common phrase associated with emotional states ("feeling a little crusty") with a literal characteristic of a pizza ("crusty"). \n\nIn everyday language, someone who is "feeling a little crusty" is usually expressing that they\'re annoyed, irritable, or experiencing stress. However, in the context of a pizza, the crust is the outer layer, typically crispy and hard. \n\nBy using this wordplay, the joke creates a pun that connects the emotional state of the pizza to its literal crust. This unexpected twist on meaning creates the humor, allowing the listener to experience a brief moment of mental processing and then arrive at the punchline, where the connection between the emotional state and the crust becomes clear.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkp

### Time Travel

In [22]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f0f6986-a0b0-6f40-8000-bfee3b1e73d5"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f0f6986-a0b0-6f40-8000-bfee3b1e73d5'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-01-21T07:11:32.305687+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f6986-a0ae-666e-bfff-f1bb2df2c2a9'}}, tasks=(PregelTask(id='9357c9d2-72e7-7864-fb63-273dd577f1c4', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty.'}),), interrupts=())

In [23]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f0f6986-a0b0-6f40-8000-bfee3b1e73d5"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty and wanted to work through some dough-y issues.',
 'explanation': "This joke plays on multiple layers of wordplay relating to pizza. \n\nFirst, the term 'crusty' is used to describe the pizza, referencing its crust. However, it's also a colloquialism used to describe someone who is irritable or short-tempered. The use of 'crusty' in this context adds a layer of wordplay, as the speaker is discussing both the literal crust of the pizza and the figurative meaning of the term.\n\nThe second layer of wordplay is the use of 'dough-y issues'. 'Dough' is a reference to the raw, uncooked mixture of flour, water, and other ingredients used to make pizza crust. However, 'issues' is a common euphemism for problems or personal struggles. The speaker is making a pun on the term 'dough' to create a clever play on words.\n\nBy combining these two forms of wordplay, the speaker is creating a clever

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy? \n\nBecause it was feeling a little crusty and wanted to work through some dough-y issues.', 'explanation': "This joke plays on multiple layers of wordplay relating to pizza. \n\nFirst, the term 'crusty' is used to describe the pizza, referencing its crust. However, it's also a colloquialism used to describe someone who is irritable or short-tempered. The use of 'crusty' in this context adds a layer of wordplay, as the speaker is discussing both the literal crust of the pizza and the figurative meaning of the term.\n\nThe second layer of wordplay is the use of 'dough-y issues'. 'Dough' is a reference to the raw, uncooked mixture of flour, water, and other ingredients used to make pizza crust. However, 'issues' is a common euphemism for problems or personal struggles. The speaker is making a pun on the term 'dough' to create a clever play on words.\n\nBy combining these two forms of wordplay, the speaker 

#### Updating State

In [25]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0f6990-541d-61c4-8000-a00690dc5d8e'}}

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc72-ca16-6359-8001-7eea05e07dd2'}}, metadata={'source': 'update', 'step': 1, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:58:35.155132+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='0f085bb0-c1e8-d9fd-fb15-c427126b7cd6', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the mushroom go to the pizza party? Because he was a fungi and everyone wanted a pizza him!', 'explanation': 'This joke plays on the word "fun guy" (fungi) which sounds like "fungi," a type of mushroom. The play on words is that the mushroom went to the pizza party because he was a "fun guy" and people 

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!',
 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa bring a ladder to the party? \nBecause it wanted to be the best snack in the room and rise to the occasion!', 'explanation': 'This joke plays on the double meaning of the word "rise." In one sense, "rise" means to physically move upwards, which is why the samosa brought a ladder to the party. However, in another sense, "rise" can also mean to perform well or excel, as in rising to the occasion. So, the samosa brought a ladder to symbolize its desire to physically rise above the other snacks at the party and also to metaphorically rise to the occasion by being the best snack in the room.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc75-4407-6195-8003-b08dcfd27511'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-29T21:59:41.628661+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoin

### Fault Tolerance

In [26]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [27]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [28]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [29]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [30]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [31]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


KeyboardInterrupt: 

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))